# 06 — 2026-10-18: full run with shuffled options and two follow-up questions, on a GPU

Experiment 05 (notebook 05) on **all 598** Emotions questions, with four changes: the options are shown in a **random
order** (position-bias control), the frames keep the videos' **native resolution** (640x360 or 854x480; earlier runs
shrank them to 448x252 for the CPU), and after the answer the model gets **two separate follow-up questions** about
the evidence it used. A new condition, `T_told`, tells the model that a video exists but is not shown. Same model revision, number of frames, prompt texts and conditions. No system message is sent, so the chat
template adds Qwen's default ("You are a helpful assistant."), as before.

| condition | input | options |
|---|---|---|
| `Q_noE` | question | A–D (floor, comparable to the pilot) |
| `Q` | question | A–E |
| `T` | transcript + question | A–E |
| `T_told` | *"A video of this scene exists, but it is not shown to you."* + T | A–E |
| `VT` | T + 64 silent frames of the question's own clip | A–E |
| `VTmis` | T + 64 silent frames of a clip from a **different film** | A–E |

**Option order.** Each question gets one random order of its five options, *"Cannot tell from the given information."*
included, so it is not always E. The same order is used in all conditions of a question (Q_noE: the same order without
"Cannot tell"), so the conditions differ only in the evidence. Results are stored under the **original** letters
(`prob_A` = the original option A, `prob_E` = "Cannot tell", `answer` = original letter), so the analysis is the same as
in notebook 05. `option_order` and `answer_shown` record what the model saw: `option_order` "CEADB" means the shown A was
the original C, the shown B was "Cannot tell", and so on.

**Follow-ups (two branches).** Each branch is the same chat (the prompt and the model's own top letter as its reply)
plus one question; neither branch sees the other, so the two are independent self-reports.

- **A, free-form:** *"Explain in one sentence which information you based your answer on."* (greedy decoding) → `explanation`
- **B, evidence question:** *"Besides the question and its options, what did you base your answer on?"* with the options
  Nothing else / A transcript / A video / Both a transcript and a video, in a random order per question (the same in
  every condition). Scored like the answer, from the letter probabilities → `evidence_prob_<option>`, `evidence_answer`.
  The question is identical in every condition, so its answers can be compared across conditions: claiming a video in
  Q or T (no video given) is claimed evidence the model never had.

Settings are in `configs/2026-10-18_full_shuffled.toml`. Results are appended to
`outputs/2026-10-18_full_shuffled/results.jsonl`; finished (question, condition) pairs are skipped, so the run can be
stopped and resumed. The model runs on the GPU when there is one. The run uses no answer keys; the analysis loads them, and no cell prints per-question keys.

In [ ]:
N_QUESTIONS = None  # None = all 598 questions
CONDITIONS = None  # None = all conditions from the config, or a subset, e.g. ["Q_noE", "Q", "T"]

In [ ]:
import sys, json, random, time, tomllib
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import qwen_omni as qo

CFG_PATH = ROOT / "configs/2026-10-18_full_shuffled.toml"
CFG = tomllib.loads(CFG_PATH.read_text())
CONDITIONS = CONDITIONS or CFG["conditions"]
assert set(CONDITIONS) <= set(CFG["conditions"]), CONDITIONS
EXP_DIR = ROOT / "outputs" / CFG["experiment"]
EXP_DIR.mkdir(parents=True, exist_ok=True)
OUT = EXP_DIR / "results.jsonl"

all_questions = pd.read_parquet(ROOT / "data-private/processed/emotion_questions.parquet")
questions = all_questions.head(N_QUESTIONS)
print(f"{len(questions)} questions, conditions {CONDITIONS} -> {OUT.relative_to(ROOT)}")

## Mismatched clips

Each question gets the clip of another Emotions question from a **different film**: the one whose clip length is
closest to its own (ties: the earlier one in the fixed order). The pairing is computed over all 598 questions, so it does
not depend on `N_QUESTIONS`. The mismatched clip uses its own frame count (64 for all but the 6 short clips). Only the
video changes: transcript, question and options are the question's own.

In [ ]:
mismatch = {}  # question_id -> the question whose clip is shown in VTmis
for q in all_questions.itertuples():
    other = all_questions[all_questions.movie_title != q.movie_title]
    gap = (other.focused_duration_s - q.focused_duration_s).abs()
    mismatch[q.question_id] = other.loc[gap.idxmin()]

pairs = pd.DataFrame({"question_id": list(mismatch),
                      "clip_s": all_questions.focused_duration_s.values,
                      "mis_question_id": [m.question_id for m in mismatch.values()],
                      "mis_movie_title": [m.movie_title for m in mismatch.values()],
                      "mis_clip_s": [m.focused_duration_s for m in mismatch.values()]})
pairs.to_csv(EXP_DIR / "mismatch_pairs.csv", index=False)
print("median |clip length difference| (s):", round((pairs.clip_s - pairs.mis_clip_s).abs().median(), 2))
pairs.head().round(1)

## Option order

One fixed random order per question (seeded by `shuffle_seed` and the question id), the same in every condition.
The count below shows how often "Cannot tell" lands on each letter; it should be roughly even. The evidence question
(branch B) gets its own random option order per question, also the same in every condition.

In [ ]:
SHOWN = ["A", "B", "C", "D", "E"]  # letters as displayed to the model


def option_order(question_id, with_e=True):
    """The question's option order: original letters in display order. with_e=False drops "Cannot tell" (Q_noE)."""
    order = ["A", "B", "C", "D", "E"]
    random.Random(f"{CFG['shuffle_seed']}-{question_id}").shuffle(order)
    return order if with_e else [x for x in order if x != "E"]


EVIDENCE = CFG["evidence_options"]  # stored name -> option text


def evidence_order(question_id):
    """The question's order of the evidence options (stored names in display order)."""
    order = list(EVIDENCE)
    random.Random(f"{CFG['shuffle_seed']}-{question_id}-evidence").shuffle(order)
    return order


def evidence_prompt(order):
    """Branch B's question, formatted like the main question."""
    options = [f"{shown}. {EVIDENCE[name]}" for shown, name in zip(SHOWN, order)]
    return CFG["evidence_question"] + "\n" + "\n".join(options) + "\n\nAnswer with the option's letter only."


e_position = pd.Series([SHOWN[option_order(qid).index("E")] for qid in all_questions.question_id])
print('"Cannot tell" shown as:', e_position.value_counts().sort_index().to_dict())

## Load the model (~18 GB RAM, a few minutes)

In [ ]:
processor, model = qo.load(CFG["model_revision"])
print("device:", model.device)

## Show the exact prompts for the first question

VT and VTmis have the same text as T, plus the video. The last two blocks are the follow-up branches A and B, shown
with "A" as a placeholder for the model's reply; in the run it is the model's own top letter.

In [ ]:
first = questions.iloc[0]
order = option_order(first.question_id)
print("option order:", "".join(order))
for title, text in [("Q_noE", qo.build_prompt(first, with_transcript=False, order=option_order(first.question_id, with_e=False))),
                    ("Q", qo.build_prompt(first, with_transcript=False, option_e=CFG["option_e"], order=order)),
                    ("T_told", CFG["told_note"] + "\n\n" + qo.build_prompt(first, option_e=CFG["option_e"], order=order)),
                    ("T, VT, VTmis", qo.build_prompt(first, option_e=CFG["option_e"], order=order))]:
    print(f"===== {title}")
    print(qo.make_inputs(processor, None, text)[1])
print("===== branch A: free-form explanation (T)")
print(qo.make_inputs(processor, None, text, followup=("A", CFG["explain_prompt"]))[1])
print("===== branch B: evidence question (T)")
print(qo.make_inputs(processor, None, text, followup=("A", evidence_prompt(evidence_order(first.question_id))))[1])

## Run

For each question and condition: score the letters, take the top one as the answer, then the two follow-up branches
(A: free-form explanation, B: evidence question), append one row.

The progress bar shows the time per question (all five conditions); times 598 gives the length of the full run.

In [ ]:
done = {(r["question_id"], r["condition"]) for r in map(json.loads, OUT.open())} if OUT.exists() else set()
print(f"{len(done)} results already in {OUT.name}")

for q in tqdm(questions.itertuples(), total=len(questions), unit="question"):
    for condition in CONDITIONS:
        if (q.question_id, condition) in done:
            continue
        t0 = time.time()

        # text: Q_noE has no option E; Q_noE and Q have no transcript; T_told starts with the note
        option_e = None if condition == "Q_noE" else CFG["option_e"]
        order = option_order(q.question_id, with_e=condition != "Q_noE")
        shown = SHOWN[:len(order)]
        user_text = qo.build_prompt(q._asdict(), with_transcript=condition in ["T", "T_told", "VT", "VTmis"],
                                    option_e=option_e, order=order)
        if condition == "T_told":
            user_text = CFG["told_note"] + "\n\n" + user_text

        # video: the question's own clip (VT) or the mismatched clip (VTmis)
        clip, frames, fps = None, None, None
        if condition == "VT":
            clip = q
        if condition == "VTmis":
            clip = mismatch[q.question_id]
        if clip is not None:
            video = ROOT / "data-private/raw/all_videos" / f"{clip.movie_title}.mp4"
            frames, fps = qo.sample_frames(video, clip.t_i, clip.t_j, int(clip.n_frames), CFG["max_pixels"])

        # 1) score the letters
        inputs, _ = qo.make_inputs(processor, None, user_text, frames, fps)
        r = qo.score_options(model, processor, inputs, shown)
        top = r["prob"].index(max(r["prob"]))
        answer_shown, answer = shown[top], order[top]  # the letter the model gave; the same option in the original order
        t1 = time.time()

        # 2) branch A: the model's answer goes into the chat, then the free-form question
        inputs, _ = qo.make_inputs(processor, None, user_text, frames, fps,
                                   followup=(answer_shown, CFG["explain_prompt"]))
        e = qo.generate_text(model, processor, inputs, CFG["explain_max_new_tokens"])
        t2 = time.time()

        # 3) branch B: the same chat up to the answer, then the evidence question, scored like the answer
        ev_order = evidence_order(q.question_id)
        inputs, _ = qo.make_inputs(processor, None, user_text, frames, fps,
                                   followup=(answer_shown, evidence_prompt(ev_order)))
        ev = qo.score_options(model, processor, inputs, SHOWN[:len(ev_order)])

        row = {"order": q.order, "question_id": q.question_id, "condition": condition,
               **{f"prob_{L}": p for L, p in zip(order, r["prob"])},  # keyed by the original letter
               "mass_on_letters": r["mass_on_letters"], "n_input_tokens": r["n_input_tokens"], "answer": answer, "answer_shown": answer_shown, "option_order": "".join(order),
               "explanation": e["text"], "explanation_tokens": e["n_tokens"], "explanation_hit_limit": e["hit_limit"],
               **{f"evidence_prob_{name}": p for name, p in zip(ev_order, ev["prob"])},
               "evidence_answer": ev_order[ev["prob"].index(max(ev["prob"]))],
               "evidence_order": ",".join(ev_order), "evidence_mass_on_letters": ev["mass_on_letters"],
               "score_seconds": round(t1 - t0, 1), "explain_seconds": round(t2 - t1, 1), "evidence_seconds": round(time.time() - t2, 1),
               "video_question_id": None if clip is None else clip.question_id,
               "n_frames": 0 if frames is None else len(frames),
               "max_pixels": CFG["max_pixels"], "model_revision": CFG["model_revision"], "device": str(model.device)}
        with OUT.open("a") as f:
            f.write(json.dumps(row) + "\n")

## Results so far

This section reads only the results file and the answer keys. To rerun it without the model, run the first two code
cells, then jump here.

- **answer** = letter with the highest probability (E = abstain)
- **accuracy** = share answered with the correct letter (an abstention counts as not correct)
- **abstain** = share where E is the top letter; **p_E** = probability on E (a softer abstention score)
- **confidence** = top probability among A–D, renormalized over A–D (comparable with the pilot)
- **selective_accuracy** = accuracy on the questions that were not abstained on
- **confidently_wrong** = wrong A–D answer with confidence ≥ `HIGH_CONFIDENCE` (the critical failure), share of all questions

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

HIGH_CONFIDENCE = 0.8
FIGURES = EXP_DIR / "figures"
FIGURES.mkdir(exist_ok=True)

keys = pd.read_parquet(ROOT / "data-private/processed/answer_keys.parquet")[["question_id", "correct_answer_key"]]
info = all_questions[["question_id", "movie_title", "question", "cue_visual_any"]]
df = pd.read_json(OUT, lines=True).drop_duplicates(["question_id", "condition"]).merge(keys, on="question_id").merge(info, on="question_id")

# keep only questions that have every condition run so far, so all conditions are compared on the same questions
n_conditions = df.groupby("question_id").condition.transform("nunique")
df = df[n_conditions == df.condition.nunique()].copy()
conditions = [c for c in CFG["conditions"] if c in set(df.condition)]
print(df.question_id.nunique(), "questions with all of", conditions)

abcd = ["prob_A", "prob_B", "prob_C", "prob_D"]
df["abstain"] = df.answer == "E"
df["p_E"] = df.prob_E.fillna(0)  # Q_noE has no option E
df["correct"] = df.answer == df.correct_answer_key
df["confidence"] = df[abcd].max(axis=1) / df[abcd].sum(axis=1)
df["confidently_wrong"] = ~df.correct & ~df.abstain & (df.confidence >= HIGH_CONFIDENCE)

In [ ]:
summary = df.groupby("condition").agg(
    n=("correct", "size"),
    accuracy=("correct", "mean"),
    abstain=("abstain", "mean"),
    mean_p_E=("p_E", "mean"),
    confidence=("confidence", "mean"),
    confidently_wrong=("confidently_wrong", "mean"),
    mass_on_letters=("mass_on_letters", "mean"),
    score_seconds=("score_seconds", "mean"),
    explain_seconds=("explain_seconds", "mean"),
    evidence_seconds=("evidence_seconds", "mean"),
).reindex(conditions)
summary.insert(3, "selective_accuracy", df[~df.abstain].groupby("condition").correct.mean())
summary.round(3)

Split by whether the annotators tagged a visual cue (face / body). The hypothesis predicts, on visual-cue questions,
more abstention without video (Q, T) than with it (VT), and more abstention with the wrong video (VTmis) than the right one.

In [ ]:
df.pivot_table(index="condition", columns="cue_visual_any", values=["correct", "abstain", "p_E", "confidently_wrong"],
               aggfunc="mean").reindex(conditions).round(3)

### Paired comparisons (same questions)

Wilcoxon signed-rank test on the per-question difference in p_E and in A–D confidence. Only meaningful with enough questions.

In [ ]:
wide = df.pivot(index="question_id", columns="condition", values=["p_E", "confidence"])
rows = []
for a, b in [("Q", "T"), ("T", "T_told"), ("T", "VT"), ("VT", "VTmis"), ("T", "VTmis")]:
    if a not in conditions or b not in conditions:
        continue
    for measure in ["p_E", "confidence"]:
        diff = wide[measure][b] - wide[measure][a]
        rows.append({"comparison": f"{b} - {a}", "measure": measure, "n": len(diff),
                     "mean_diff": diff.mean(), "median_diff": diff.median(),
                     "p": wilcoxon(diff).pvalue if (diff != 0).any() else None})
pd.DataFrame(rows).round(4)

### Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, col, title in [(axes[0], "accuracy", "Accuracy"),
                       (axes[1], "abstain", "Abstained (E is the top letter)"),
                       (axes[2], "confidently_wrong", f"Wrong with confidence ≥ {HIGH_CONFIDENCE}")]:
    ax.bar(summary.index, summary[col], color="tab:blue")
    for i, value in enumerate(summary[col]):
        ax.text(i, value + 0.01, f"{value:.0%}", ha="center")
    ax.set(title=title, ylim=(0, 1))
axes[0].axhline(0.25, ls="--", c="gray")
fig.suptitle(f"{df.question_id.nunique()} emotion questions")
fig.tight_layout()
fig.savefig(FIGURES / "summary_by_condition.png", dpi=200)
plt.show()

### More plots

Shared style and helpers. Colors: blue for a single series; blue and orange when two groups are compared.
Error bars are 95% intervals: bootstrap over questions for means, Wilson for proportions.

In [ ]:
import numpy as np

BLUE, ORANGE, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e4e2dc"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": MUTED,
                     "axes.labelcolor": INK, "axes.titlecolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "axes.axisbelow": True,
                     "font.size": 11, "legend.frameon": False})

def bootstrap_ci(values, n_resamples=2000, seed=0):
    """95% bootstrap interval for the mean (resampling questions)."""
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    means = [rng.choice(values, len(values)).mean() for _ in range(n_resamples)]
    return np.percentile(means, [2.5, 97.5])

def wilson(k, n, z=1.96):
    """95% interval for a proportion k/n."""
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * (p * (1 - p) / n + z**2 / (4 * n**2)) ** 0.5 / (1 + z**2 / n)
    return centre - half, centre + half

# extra columns used by the plots
df["forced_answer"] = df[abcd].idxmax(axis=1).str[-1]  # best of A-D, ignoring E
df["forced_correct"] = df.forced_answer == df.correct_answer_key
df["no_dialogue"] = df.question_id.map(all_questions.set_index("question_id").focused_n_words) == 0
df["outcome"] = np.where(df.abstain, "abstain", np.where(df.correct, "right", "wrong"))
with_e = [c for c in conditions if c != "Q_noE"]

#### 1. Abstention follows the evidence

Mean P(E) and the share of questions where E is the top letter, from the least evidence (question only) to the most
(question + transcript + the right video), and with the wrong video.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, title in zip(axes, ["Mean probability on E", "Abstained (E is the top letter)"]):
    for i, condition in enumerate(with_e):
        g = df[df.condition == condition]
        if title.startswith("Mean"):
            value, (lo, hi) = g.p_E.mean(), bootstrap_ci(g.p_E)
        else:
            value, (lo, hi) = g.abstain.mean(), wilson(g.abstain.sum(), len(g))
        ax.errorbar(i, value, yerr=[[value - lo], [hi - value]], fmt="o", ms=9, color=BLUE, capsize=6, lw=2)
        ax.text(i + 0.12, value, f"{value:.2f}" if title.startswith("Mean") else f"{value:.0%}", va="center", color=INK)
    ax.set(xticks=range(len(with_e)), xticklabels=with_e, ylim=(0, 1), xlim=(-0.4, len(with_e) - 0.4), title=title)
fig.suptitle(f"Abstention by condition ({df.question_id.nunique()} questions)")
fig.tight_layout()
fig.savefig(FIGURES / "abstention_by_condition.png", dpi=200)
plt.show()

#### 2. The video changes *whether* the model answers, not *what* it answers

Each bar splits all questions into four outcomes. Solid: the model answered (right or wrong). Hatched: it abstained;
the hatch color shows whether its best A–D option would have been right or wrong. If the video gave better evidence,
VT would have more solid blue than T *and* a better ratio of blue to orange among the answered questions.

In [ ]:
parts = [("answered, right", ~df.abstain & df.correct, BLUE, None),
         ("answered, wrong", ~df.abstain & ~df.correct, ORANGE, None),
         ("abstained, best A–D was right", df.abstain & df.forced_correct, BLUE, "///"),
         ("abstained, best A–D was wrong", df.abstain & ~df.forced_correct, ORANGE, "///")]

fig, ax = plt.subplots(figsize=(11, 4.2))
rows = np.arange(len(conditions))
left = np.zeros(len(conditions))
for label, mask, color, hatch in parts:
    share = mask.groupby(df.condition).mean().reindex(conditions).values
    hatched = hatch is not None
    keep = share > 0  # skip empty segments (e.g. no abstentions in Q_noE)
    ax.barh(rows[keep], share[keep], left=left[keep], color="white" if hatched else color, edgecolor=color if hatched else "white",
            hatch=hatch, linewidth=1.5 if hatched else 2, label=label, height=0.6)
    for y, (x, w) in enumerate(zip(left, share)):
        if w >= 0.06:
            ax.text(x + w / 2, y, f"{w:.0%}", ha="center", va="center", color=INK if hatched else "white",
                    fontweight="bold", bbox=dict(facecolor="white", edgecolor="none", pad=1) if hatched else None)
    left += share
ax.set_yticks(rows, conditions)
ax.invert_yaxis()
ax.set(xlim=(0, 1), xlabel="share of questions", title="Outcome of every question, by condition")
ax.grid(axis="y", visible=False)
ax.grid(axis="x", color=GRID)
ax.legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.18))
fig.tight_layout()
fig.savefig(FIGURES / "outcomes_by_condition.png", dpi=200)
plt.show()

print("accuracy among answered:", df[~df.abstain].groupby("condition").correct.mean().reindex(conditions).round(3).to_dict())
print("forced A-D accuracy:   ", df.groupby("condition").forced_correct.mean().reindex(conditions).round(3).to_dict())

#### 3. Question by question: P(E) with and without the (right) video

One dot per question. Above the diagonal: more abstention in the second condition. Left: adding the video (T → VT).
Right: swapping in the wrong video (VT → VTmis).

In [ ]:
pE = df.pivot(index="question_id", columns="condition", values="p_E")
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, (a, b) in zip(axes, [("T", "VT"), ("VT", "VTmis")]):
    ax.plot([0, 1], [0, 1], color=MUTED, lw=1, ls="--")
    ax.scatter(pE[a], pE[b], s=40, color=BLUE, alpha=0.7, edgecolor="white", linewidth=1)
    up, down = (pE[b] > pE[a] + 0.01).sum(), (pE[b] < pE[a] - 0.01).sum()
    ax.text(0.03, 0.95, f"higher in {b}: {up}", transform=ax.transAxes, va="top", color=INK, bbox=dict(facecolor="white", edgecolor=GRID))
    ax.text(0.97, 0.05, f"lower in {b}: {down}", transform=ax.transAxes, ha="right", color=INK, bbox=dict(facecolor="white", edgecolor=GRID))
    ax.set(xlim=(0, 1), ylim=(0, 1), aspect="equal", xlabel=f"P(E) in {a}", ylabel=f"P(E) in {b}", title=f"{a} → {b}")
    ax.grid(axis="both", color=GRID)
fig.tight_layout()
fig.savefig(FIGURES / "p_E_per_question.png", dpi=200)
plt.show()

#### 4. Where the answers move

Counts of questions by outcome in one condition (rows) and another (columns). The diagonal is "no change".

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

blues = LinearSegmentedColormap.from_list("blues", ["#f5f9fe", "#9ec5f4", "#2a78d6", "#104281"])
outcome = df.pivot(index="question_id", columns="condition", values="outcome")
labels = ["right", "wrong", "abstain"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, (a, b) in zip(axes, [("Q", "T"), ("T", "VT"), ("VT", "VTmis")]):
    table = pd.crosstab(outcome[a], outcome[b]).reindex(index=labels, columns=labels, fill_value=0)
    ax.imshow(table.values, cmap=blues, vmin=0, vmax=table.values.max())
    for i in range(3):
        for j in range(3):
            n = table.values[i, j]
            ax.text(j, i, n, ha="center", va="center", fontsize=13,
                    color="white" if n > table.values.max() * 0.55 else INK)
    ax.set(xticks=range(3), xticklabels=labels, yticks=range(3), yticklabels=labels,
           xlabel=b, ylabel=a, title=f"{a} → {b}")
    ax.grid(False)
    ax.spines[:].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURES / "outcome_transitions.png", dpi=200)
plt.show()

#### 5. Subgroups: visual-cue tag and clips without dialogue

Abstention rate across conditions for each subgroup. Groups are small (see n in the labels), so read these as
patterns, not results.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)
for ax, (column, names) in zip(axes, [("cue_visual_any", {True: "visual cue tagged", False: "no visual cue tag"}),
                                      ("no_dialogue", {False: "clip has dialogue", True: "no dialogue in clip"})]):
    for value, color in [(True, BLUE), (False, ORANGE)]:
        g = df[(df[column] == value) & df.condition.isin(with_e)]
        rate = g.groupby("condition").abstain.mean().reindex(with_e)
        n = g.question_id.nunique()
        ax.plot(range(len(with_e)), rate.values, "o-", color=color, lw=2, ms=8, label=f"{names[value]} (n={n})")
        ax.text(len(with_e) - 1 + 0.1, rate.values[-1], f"{names[value]}\nn={n}", va="center", color=INK, fontsize=9)
    ax.set(xticks=range(len(with_e)), xticklabels=with_e, ylim=(0, 1), xlim=(-0.3, len(with_e) + 0.6),
           title=f"Abstention by {'visual-cue tag' if column == 'cue_visual_any' else 'dialogue'}")
    ax.legend(loc="lower left")
axes[0].set_ylabel("abstained (E is the top letter)")
fig.tight_layout()
fig.savefig(FIGURES / "abstention_subgroups.png", dpi=200)
plt.show()

## Explanations

One row per question and condition, also saved as `explanations.csv` in the experiment folder for reading or hand coding.
No answer keys are saved here or shown below (only aggregate scores use them), so the file can be in git.

In [ ]:
explanations = df[["order", "question_id", "movie_title", "question", "condition", "option_order", "answer_shown", "answer",
                   "confidence", "p_E", "explanation", "evidence_answer", "explanation_hit_limit"]].sort_values(["order", "condition"])
explanations.to_csv(EXP_DIR / "explanations.csv", index=False)
print("explanations that hit the token limit:", explanations.explanation_hit_limit.sum())
pd.set_option("display.max_colwidth", None)
explanations.head(15)

### First look: does the model claim visual evidence it did not have?

A crude keyword screen for words about faces, expressions or body language. Such words in **Q / T** (no video) or in
**VTmis** (video from another film) describe evidence the model did not have. This is only a quick look; the real
analysis needs the explanations coded by hand (or by an LLM judge).

In [ ]:
VISUAL_WORDS = ["face", "facial", "expression", "smile", "smiling", "frown", "eyes", "gaze", "tears", "crying",
                "body language", "posture", "gesture", "video", "visible"]
pattern = r"\b(?:" + "|".join(VISUAL_WORDS) + r")\b"
explanations["mentions_visual"] = explanations.explanation.str.contains(pattern, case=False)
display(explanations.groupby("condition").mentions_visual.mean().reindex(conditions).round(3))
explanations[(explanations.condition == "VTmis") & explanations.mentions_visual & (explanations.answer != "E")].head(10)

#### 6. What the explanations talk about

Share of explanations per condition that (a) use words about faces, expressions or body language, (b) refer to the
transcript or dialogue, (c) say the information is not enough. Keyword screens only; (a) in Q_noE, Q and T is
visual evidence the model never had.

In [ ]:
screens = {
    "visual words": r"\b(?:face|facial|expression|smil\w*|frown\w*|eyes|gaze|tears|cry\w*|body language|posture|gesture\w*)\b",
    "transcript / dialogue": r"\b(?:transcript|dialogue|says|said)\b",
    "'not enough information'": r"does not provide|not enough|cannot be determined|impossible to determine|no (?:information|dialogue|context)",
}
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, (title, pattern) in zip(axes, screens.items()):
    share = explanations.explanation.str.contains(pattern, case=False).groupby(explanations.condition).mean().reindex(conditions)
    ax.bar(conditions, share.values, color=BLUE, width=0.6)
    for i, v in enumerate(share.values):
        ax.text(i, v + 0.015, f"{v:.0%}", ha="center", color=INK)
    ax.set(title=title, ylim=(0, 1))
axes[0].set_ylabel("share of explanations")
fig.tight_layout()
fig.savefig(FIGURES / "explanation_screens.png", dpi=200)
plt.show()

## Told that a video exists (T_told vs T)

Does the model hold back more when it is told that a video exists but is not shown? A model sensitive to evidence
should abstain more in T_told mainly on the questions where the video matters, not on all questions. As a proxy, the
video "mattered" when the answer in VT differs from the answer in T. A rise of the same size in both groups is a
reaction to the sentence, not to the missing evidence.

In [ ]:
if {"T", "T_told", "VT"} <= set(conditions):
    answers = df.pivot(index="question_id", columns="condition", values="answer")
    video_mattered = (answers["VT"] != answers["T"]).rename("video_mattered")
    told = df[df.condition.isin(["T", "T_told"])].merge(video_mattered, left_on="question_id", right_index=True)
    display(told.pivot_table(index="video_mattered", columns="condition", values=["abstain", "p_E"], aggfunc="mean").round(3))
    print("questions per group:", video_mattered.value_counts().to_dict())

## Evidence question (branch B)

Share of each answer to *"Besides the question and its options, what did you base your answer on?"*, per condition.
What the model actually had: Q_noE and Q nothing else, T and T_told a transcript, VT and VTmis both. A video (or both)
claimed in Q_noE, Q, T or T_told is evidence it never had (in T_told, right after being told the video is not shown). The second table splits by whether the model abstained, since after an
abstention the question reads as "what led you to abstain". `evidence_mass_on_letters` checks that the model answered
with a letter at all.

In [ ]:
evidence_names = list(EVIDENCE)
display(pd.crosstab(df.condition, df.evidence_answer, normalize="index")
        .reindex(index=conditions, columns=evidence_names, fill_value=0).round(3))
display(pd.crosstab([df.condition, df.abstain], df.evidence_answer, normalize="index")
        .reindex(columns=evidence_names, fill_value=0).round(3))
claims_video = df.evidence_answer.isin(["video", "both"])
print("claimed a video:", claims_video.groupby(df.condition).mean().reindex(conditions).round(3).to_dict())
print("mean evidence_mass_on_letters:", round(df.evidence_mass_on_letters.mean(), 3))

## Option position

Does the letter an option is shown as matter? Left: abstention rate by the letter "Cannot tell" was shown as. Without
position bias the rates are similar across letters. Right: share of answers by the letter given.

In [ ]:
e_conditions = [c for c in conditions if c != "Q_noE"]
df["e_shown_as"] = df.option_order.str.find("E").map(dict(enumerate(SHOWN)))  # NaN for Q_noE
display(df[df.condition.isin(e_conditions)]
        .pivot_table(index="condition", columns="e_shown_as", values="abstain", aggfunc="mean")
        .reindex(e_conditions).round(3))
display(pd.crosstab(df.condition, df.answer_shown, normalize="index").reindex(conditions).round(3))

## Run time

In [ ]:
seconds = df.score_seconds + df.explain_seconds + df.evidence_seconds
display(seconds.groupby(df.condition).mean().reindex(conditions).round(1).rename("seconds per question"))
print(f"{seconds.sum() / 3600:.1f} GPU-hours for {df.question_id.nunique()} questions (model loading not included)")